In [1]:
import torch
from torch import nn

class SmallNetwork(nn.Module):
    def __init__(self, l2i, l3i, l4i):
        super().__init__()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(3, l2i),
            nn.ReLU(),
            nn.Linear(l2i, l3i),
            nn.ReLU(),
            nn.Linear(l3i, l4i),
            nn.ReLU(),
            nn.Linear(l4i, 1),
        )

    def forward(self, x):
        return self.linear_relu_stack(x)

model = SmallNetwork(3,3,3)

# inspect weights
for name, param in model.named_parameters():
    print(name, param.shape, '\n', param.data)


linear_relu_stack.0.weight torch.Size([3, 3]) 
 tensor([[ 0.3815,  0.4832, -0.3245],
        [ 0.2819,  0.3833,  0.0267],
        [-0.1873, -0.2745, -0.4074]])
linear_relu_stack.0.bias torch.Size([3]) 
 tensor([ 0.3335,  0.3400, -0.1219])
linear_relu_stack.2.weight torch.Size([3, 3]) 
 tensor([[ 0.5096,  0.4143, -0.5218],
        [ 0.0842, -0.1712, -0.3680],
        [ 0.2935, -0.1605, -0.2206]])
linear_relu_stack.2.bias torch.Size([3]) 
 tensor([ 0.1622, -0.3778, -0.4372])
linear_relu_stack.4.weight torch.Size([3, 3]) 
 tensor([[-0.0723,  0.5324, -0.3518],
        [-0.4629,  0.5545, -0.4547],
        [ 0.3459,  0.3054,  0.4773]])
linear_relu_stack.4.bias torch.Size([3]) 
 tensor([-0.2461,  0.4394,  0.2703])
linear_relu_stack.6.weight torch.Size([1, 3]) 
 tensor([[-0.2725,  0.4818,  0.2879]])
linear_relu_stack.6.bias torch.Size([1]) 
 tensor([-0.1850])


In [2]:
linear_layers = [(i, m) for i, m in enumerate(model.linear_relu_stack) if isinstance(m, nn.Linear)]

saved = [[layer.weight.data.clone(), layer.bias.data.clone()] 
         for _, layer in linear_layers]


In [22]:
def remove(saved, layer, node):
    # remove row from current layer weight
    saved[layer][0] = torch.cat([saved[layer][0][:node], saved[layer][0][node+1:]], dim=0)
    # remove element from current layer bias
    saved[layer][1] = torch.cat([saved[layer][1][:node], saved[layer][1][node+1:]])
    # remove column from next layer weight
    saved[layer+1][0] = torch.cat([saved[layer+1][0][:, :node], saved[layer+1][0][:, node+1:]], dim=1)

def leastweight(saved):
    layer = 0
    node = 0
    v = torch.inf
    for l in range(len(saved)-1):
        importance = saved[l][0].abs().sum(dim=0)
        print(importance)
        if v >= min(importance).item():
            v = min(importance).item()
            layer = l
            node = importance.argmin().item()
    return layer, node

In [24]:
leastweight(saved)


tensor([0.8507, 1.1411, 0.7587])
tensor([0.8873, 0.7460, 1.1105])
tensor([0.8811, 1.3922, 1.2839])


(1, 1)

In [52]:
remove(saved, 1, 1)

In [53]:
l2i = saved[0][0].shape[0]  # out_features of layer 0
l3i = saved[1][0].shape[0]  # out_features of layer 1
l4i = saved[2][0].shape[0]  # out_features of layer 2

new_model = SmallNetwork(l2i, l3i, l4i)
new_linear_layers = [(i, m) for i, m in enumerate(new_model.linear_relu_stack) if isinstance(m, nn.Linear)]

for (w, b), (_, layer) in zip(saved, new_linear_layers):
    layer.weight.data = w.clone()
    layer.bias.data   = b.clone()

In [ ]:
for name, param in new_model.named_parameters():
    print(name, param.shape, '\n', param.data)

linear_relu_stack.0.weight torch.Size([3, 3]) 
 tensor([[ 0.0841, -0.0867, -0.3572],
        [-0.5328, -0.1400,  0.1248],
        [ 0.4014,  0.0816,  0.2345]])
linear_relu_stack.0.bias torch.Size([3]) 
 tensor([-0.0687, -0.3328, -0.4373])
linear_relu_stack.2.weight torch.Size([2, 3]) 
 tensor([[ 0.5419,  0.4346, -0.0831],
        [-0.3588,  0.2667,  0.0885]])
linear_relu_stack.2.bias torch.Size([2]) 
 tensor([-0.4896, -0.0023])
linear_relu_stack.4.weight torch.Size([3, 2]) 
 tensor([[-0.4442, -0.0715],
        [ 0.3038,  0.3071],
        [ 0.1273, -0.0410]])
linear_relu_stack.4.bias torch.Size([3]) 
 tensor([-0.0715, -0.5560, -0.0438])
linear_relu_stack.6.weight torch.Size([1, 3]) 
 tensor([[-0.4606, -0.4527,  0.1731]])
linear_relu_stack.6.bias torch.Size([1]) 
 tensor([0.2065])
